# 🍎 AlphaApple Training V5 (PPO Fine-tuning)

**V5 핵심 개선사항**: BC → PPO (진짜 RL!)

## 🎯 V4 문제점 분석
- **V4 결과**: 평균 102.4개 (60.2%)
- **방법론**: Behavior Cloning (BC)
- **문제**: BC는 expert를 모방만 함 → expert 넘을 수 없음!

## 💡 V5 해결책: PPO (Proximal Policy Optimization)

**핵심 차이점**:

```
BC (V1-V4):           →  PPO (V5):
- Expert 모방           - Reward 최대화
- Ceiling 존재          - Expert 초과 가능
- Exploration X         - Exploration O
```

**PPO 알고리즘**:
1. Policy로 환경 플레이 (rollouts)
2. Advantage 계산 (GAE)
3. Clipped objective로 policy 업데이트
4. Value function 업데이트

**Warm-start**:
- BC 모델에서 시작 (bc_policy_best_v4.pt)
- PPO로 fine-tuning
- 안정적 학습 보장

## 🎯 최종 목표
**V4**: 102개 → **V5**: 110-120개 (첫 PPO 시도)

**궁극적 목표**: 170점 만점 (100% 클리어) 🍎

## 🔧 Setup

In [ ]:
# Colab 환경 확인
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Colab")
except:
    IN_COLAB = False
    print("❌ Not in Colab")

# GPU 확인
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# GitHub에서 코드 가져오기
if IN_COLAB:
    !git clone https://github.com/kbsooo/AlphaApple.git
    %cd AlphaApple
    !git checkout claude/alphaapple-v4-training-results-011CV3HHKYhWC9t7CCeCBhmn
else:
    import os
    os.chdir('/home/user/AlphaApple')

In [ ]:
# 의존성 설치
!pip install -q gymnasium numpy torch tqdm

## 📦 Import & BC 모델 로드

In [ ]:
import sys
import numpy as np
import torch
from tqdm.notebook import tqdm

sys.path.insert(0, '.')

from models.lightweight_policy import LightweightPolicy
from envs.autoregressive_wrapper import make_autoregressive_env
from envs.backward_generator import BackwardBoardGenerator
from algorithms.ppo import PPOBuffer, PPOTrainer, collect_rollouts

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

In [ ]:
# BC 모델 로드 (V4에서 학습된 모델)
print("=== BC 모델 로드 (warm-start) ===")

policy = LightweightPolicy(rows=10, cols=17, latent_dim=128)

# V4 best 모델이 있다면 로드, 없으면 랜덤 초기화
import os
if os.path.exists('bc_policy_best_v4.pt'):
    policy.load_state_dict(torch.load('bc_policy_best_v4.pt', map_location='cpu'))
    print("✅ BC 모델 로드 완료: bc_policy_best_v4.pt")
    print("   (V4 성능: 평균 102.4개)")
else:
    print("⚠️  BC 모델 파일 없음. 랜덤 초기화로 시작합니다.")
    print("   (권장: V4 먼저 실행해서 bc_policy_best_v4.pt 생성)")

policy = policy.to(device)

total_params = sum(p.numel() for p in policy.parameters())
print(f"\n모델 파라미터 수: {total_params:,}")
print()

## 📊 BC 모델 평가 (Before PPO)

In [ ]:
def evaluate_policy(policy, n_episodes=20, target_coverage=0.95, device='cuda', seed_offset=0):
    """
    정책 평가
    """
    episode_rewards = []
    
    policy.eval()
    with torch.no_grad():
        for i in range(n_episodes):
            generator = BackwardBoardGenerator(rows=10, cols=17, seed=seed_offset + i)
            board, _ = generator.generate(target_coverage=target_coverage)
            
            wrapped_env = make_autoregressive_env(rows=10, cols=17)
            env = wrapped_env.env
            env.board = board.astype(np.int16)
            obs = board.clip(0, 9).astype(np.int8)
            
            episode_reward = 0
            steps = 0
            
            while steps < 500:
                masks_np = wrapped_env.get_autoregressive_masks()
                masks_torch = {
                    'r1_mask': torch.from_numpy(masks_np['r1_mask']).to(device),
                    'c1_masks': torch.from_numpy(masks_np['c1_masks']).to(device),
                    'r2_masks': torch.from_numpy(masks_np['r2_masks']).to(device),
                    'c2_masks': torch.from_numpy(masks_np['c2_masks']).to(device)
                }
                
                obs_tensor = torch.from_numpy(obs).float().unsqueeze(0).unsqueeze(0).to(device)
                action_tuple, _, _, _ = policy(obs_tensor, deterministic=True, masks=masks_torch)
                
                r1 = int(action_tuple[0][0].item())
                c1 = int(action_tuple[1][0].item())
                r2 = int(action_tuple[2][0].item())
                c2 = int(action_tuple[3][0].item())
                
                obs, reward, terminated, truncated, info = wrapped_env.step_with_coords(r1, c1, r2, c2)
                
                episode_reward += reward
                steps += 1
                
                if terminated or truncated:
                    break
            
            episode_rewards.append(episode_reward)
    
    return episode_rewards


print("=== BC 모델 평가 (Before PPO) ===")
bc_rewards = evaluate_policy(policy, n_episodes=30, device=device, seed_offset=10000)

print(f"평균: {np.mean(bc_rewards):.1f} ± {np.std(bc_rewards):.1f}")
print(f"최대: {max(bc_rewards):.0f}/170 ({max(bc_rewards)/170*100:.1f}%)")
print(f"최소: {min(bc_rewards):.0f}")
print()

## 🚀 PPO 학습

In [ ]:
# PPO 하이퍼파라미터
PPO_CONFIG = {
    # Rollout
    'n_steps': 2048,          # Rollout 길이
    'gamma': 0.99,            # Discount factor
    'gae_lambda': 0.95,       # GAE lambda
    
    # Training
    'n_epochs': 4,            # PPO epochs per update
    'batch_size': 64,         # Mini-batch size
    'lr': 1e-4,               # Learning rate (BC보다 낮음)
    
    # PPO specific
    'clip_range': 0.2,        # PPO clip range
    'value_coef': 0.5,        # Value loss coefficient
    'entropy_coef': 0.01,     # Entropy bonus
    'max_grad_norm': 0.5,     # Gradient clipping
    
    # Overall
    'total_timesteps': 50000, # 총 학습 스텝
    'eval_freq': 10000,       # 평가 빈도
}

print("=== PPO 설정 ===")
for key, value in PPO_CONFIG.items():
    print(f"  {key}: {value}")
print()

In [ ]:
# PPO Trainer 초기화
trainer = PPOTrainer(
    policy=policy,
    lr=PPO_CONFIG['lr'],
    clip_range=PPO_CONFIG['clip_range'],
    value_coef=PPO_CONFIG['value_coef'],
    entropy_coef=PPO_CONFIG['entropy_coef'],
    max_grad_norm=PPO_CONFIG['max_grad_norm'],
    device=device
)

# Environment
env_wrapper = make_autoregressive_env(rows=10, cols=17)

print("✅ PPO Trainer 초기화 완료")
print()

In [ ]:
# PPO 학습 루프
print("="*70)
print("🍎 AlphaApple V5: PPO Fine-tuning")
print("="*70)
print(f"Total timesteps: {PPO_CONFIG['total_timesteps']:,}")
print(f"Rollout length: {PPO_CONFIG['n_steps']:,}")
print(f"Updates: {PPO_CONFIG['total_timesteps'] // PPO_CONFIG['n_steps']}")
print("="*70)
print()

timesteps = 0
update = 0
seed_offset = 0

training_history = {
    'timesteps': [],
    'mean_reward': [],
    'policy_loss': [],
    'value_loss': [],
    'entropy': [],
    'eval_rewards': [],
}

while timesteps < PPO_CONFIG['total_timesteps']:
    update += 1
    
    print(f"\n{'='*70}")
    print(f"Update {update} | Timesteps: {timesteps}/{PPO_CONFIG['total_timesteps']}")
    print(f"{'='*70}")
    
    # Collect rollouts
    buffer = PPOBuffer(gamma=PPO_CONFIG['gamma'], gae_lambda=PPO_CONFIG['gae_lambda'])
    
    print("\n[1] Collecting rollouts...")
    rollout_info = collect_rollouts(
        policy=policy,
        env_wrapper=env_wrapper,
        buffer=buffer,
        n_steps=PPO_CONFIG['n_steps'],
        device=device,
        gamma=PPO_CONFIG['gamma'],
        seed_offset=seed_offset
    )
    
    timesteps += PPO_CONFIG['n_steps']
    seed_offset += rollout_info['n_episodes']
    
    print(f"  Episodes: {rollout_info['n_episodes']}")
    print(f"  Mean reward: {rollout_info['mean_episode_reward']:.1f}")
    print(f"  Mean length: {rollout_info['mean_episode_length']:.1f}")
    
    # PPO update
    print("\n[2] PPO update...")
    policy.train()
    update_info = trainer.update(
        buffer=buffer,
        n_epochs=PPO_CONFIG['n_epochs'],
        batch_size=PPO_CONFIG['batch_size']
    )
    
    print(f"  Policy loss: {update_info['policy_loss']:.4f}")
    print(f"  Value loss: {update_info['value_loss']:.4f}")
    print(f"  Entropy: {update_info['entropy']:.4f}")
    print(f"  Clip fraction: {update_info['clip_fraction']:.3f}")
    print(f"  Approx KL: {update_info['approx_kl']:.4f}")
    
    # Save history
    training_history['timesteps'].append(timesteps)
    training_history['mean_reward'].append(rollout_info['mean_episode_reward'])
    training_history['policy_loss'].append(update_info['policy_loss'])
    training_history['value_loss'].append(update_info['value_loss'])
    training_history['entropy'].append(update_info['entropy'])
    
    # Evaluation
    if timesteps % PPO_CONFIG['eval_freq'] == 0 or timesteps >= PPO_CONFIG['total_timesteps']:
        print("\n[3] Evaluating...")
        eval_rewards = evaluate_policy(policy, n_episodes=20, device=device, seed_offset=20000+update)
        eval_mean = np.mean(eval_rewards)
        eval_max = max(eval_rewards)
        
        print(f"  Eval mean: {eval_mean:.1f}")
        print(f"  Eval max: {eval_max:.0f}")
        
        training_history['eval_rewards'].append((timesteps, eval_mean, eval_max))
        
        # Save checkpoint
        torch.save(policy.state_dict(), f'ppo_policy_v5_{timesteps}.pt')
        print(f"  Model saved: ppo_policy_v5_{timesteps}.pt")

print("\n" + "="*70)
print("🎉 PPO 학습 완료!")
print("="*70)

## 📊 최종 평가 (50 episodes)

In [ ]:
print("=== 최종 평가 (50 episodes) ===")
final_rewards = evaluate_policy(policy, n_episodes=50, device=device, seed_offset=30000)

print(f"평균: {np.mean(final_rewards):.1f} ± {np.std(final_rewards):.1f}")
print(f"최대: {max(final_rewards):.0f}/170 ({max(final_rewards)/170*100:.1f}%)")
print(f"최소: {min(final_rewards):.0f}")
print(f"범위: [{min(final_rewards):.0f}, {max(final_rewards):.0f}]")
print()

# 최종 모델 저장
torch.save(policy.state_dict(), 'ppo_policy_best_v5.pt')
print("✅ 최종 모델 저장: ppo_policy_best_v5.pt")

## 📊 결과 요약 (Claude Code용)

In [ ]:
print("\n" + "="*70)
print("🍎 AlphaApple V5 Training Summary (Claude Code용)")
print("="*70)
print()
print("[1] PPO 설정")
print(f"  - Total timesteps: {PPO_CONFIG['total_timesteps']:,}")
print(f"  - Learning rate: {PPO_CONFIG['lr']}")
print(f"  - Clip range: {PPO_CONFIG['clip_range']}")
print()
print("[2] 학습 결과")
print(f"  - BC 시작: {np.mean(bc_rewards):.1f}개")
print(f"  - PPO 종료: {np.mean(final_rewards):.1f}개")
print(f"  - 개선량: +{np.mean(final_rewards) - np.mean(bc_rewards):.1f}개")
print()
print("[3] 최종 평가 결과 (50 episodes)")
print(f"  평균: {np.mean(final_rewards):.1f}개 ({np.mean(final_rewards)/170*100:.1f}%)")
print(f"  최대: {max(final_rewards):.0f}개 ({max(final_rewards)/170*100:.1f}%)")
print(f"  범위: [{min(final_rewards):.0f}, {max(final_rewards):.0f}]")
print()
print("[4] 버전 비교")
print("  | 버전 | 방법 | 평균 (95%) | 최대 |")
print("  |------|------|-----------|------|")
print("  | V1   | BC (no mask) | -500 (0%) | 0    |")
print("  | V2   | BC + Mask | 101.5 (59.7%) | 129  |")
print("  | V3   | Multi-Rollout | 101.9 (59.9%) | 120  |")
print("  | V4   | Expert Iter | 102.4 (60.2%) | 125  |")
print(f"  | V5   | PPO | {np.mean(final_rewards):.1f} ({np.mean(final_rewards)/170*100:.1f}%) | {max(final_rewards):.0f}  |")
print()
print("[5] 목표 대비 (최종 목표: 170점 만점)")
print(f"  V5 최고:   {max(final_rewards):.0f}개 ({max(final_rewards)/170*100:.1f}%)")
print(f"  목표까지:  {170 - max(final_rewards):.0f}개 남음")
print(f"  진척률:    {max(final_rewards)/170*100:.1f}%")
print()
print("[6] PPO 효과")
improvement_pct = (np.mean(final_rewards) - np.mean(bc_rewards)) / np.mean(bc_rewards) * 100
print(f"  BC (Before): {np.mean(bc_rewards):.1f}개")
print(f"  PPO (After): {np.mean(final_rewards):.1f}개")
print(f"  개선률:      {improvement_pct:+.1f}%")
print()
print("[7] 다음 단계 제안")
if max(final_rewards) >= 140:
    print("  🎯 거의 도달! MCTS로 완벽 플레이 시도")
elif max(final_rewards) >= 120:
    print("  📈 좋은 진전! 모델 크기 늘려서 추가 개선 시도")
elif max(final_rewards) > np.mean(bc_rewards) + 5:
    print("  ✅ PPO 효과 확인! 더 많은 timesteps 또는 모델 확대")
else:
    print("  🔄 하이퍼파라미터 튜닝 또는 reward shaping 필요")
print()
print("="*70)

## 📈 시각화

In [ ]:
import matplotlib.pyplot as plt

# Training curve
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Mean reward
axes[0, 0].plot(training_history['timesteps'], training_history['mean_reward'], linewidth=2)
axes[0, 0].axhline(y=np.mean(bc_rewards), color='r', linestyle='--', label=f'BC baseline ({np.mean(bc_rewards):.1f})')
axes[0, 0].set_xlabel('Timesteps')
axes[0, 0].set_ylabel('Mean Episode Reward')
axes[0, 0].set_title('Training Progress')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Policy loss
axes[0, 1].plot(training_history['timesteps'], training_history['policy_loss'], linewidth=2, color='orange')
axes[0, 1].set_xlabel('Timesteps')
axes[0, 1].set_ylabel('Policy Loss')
axes[0, 1].set_title('Policy Loss')
axes[0, 1].grid(True, alpha=0.3)

# Value loss
axes[1, 0].plot(training_history['timesteps'], training_history['value_loss'], linewidth=2, color='green')
axes[1, 0].set_xlabel('Timesteps')
axes[1, 0].set_ylabel('Value Loss')
axes[1, 0].set_title('Value Loss')
axes[1, 0].grid(True, alpha=0.3)

# Entropy
axes[1, 1].plot(training_history['timesteps'], training_history['entropy'], linewidth=2, color='purple')
axes[1, 1].set_xlabel('Timesteps')
axes[1, 1].set_ylabel('Entropy')
axes[1, 1].set_title('Policy Entropy')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v5_ppo_training.png', dpi=150)
plt.show()

print("✅ 그래프 저장: v5_ppo_training.png")

# Evaluation plot
if training_history['eval_rewards']:
    eval_timesteps = [x[0] for x in training_history['eval_rewards']]
    eval_means = [x[1] for x in training_history['eval_rewards']]
    eval_maxs = [x[2] for x in training_history['eval_rewards']]
    
    plt.figure(figsize=(10, 6))
    plt.plot(eval_timesteps, eval_means, marker='o', label='Mean', linewidth=2)
    plt.plot(eval_timesteps, eval_maxs, marker='s', label='Max', linewidth=2)
    plt.axhline(y=np.mean(bc_rewards), color='r', linestyle='--', label=f'BC baseline ({np.mean(bc_rewards):.1f})')
    plt.axhline(y=170, color='g', linestyle='--', label='Perfect (170)', alpha=0.5)
    plt.xlabel('Timesteps', fontsize=12)
    plt.ylabel('Reward (cells removed)', fontsize=12)
    plt.title('V5: PPO Evaluation Progress', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('v5_ppo_eval.png', dpi=150)
    plt.show()
    
    print("✅ 그래프 저장: v5_ppo_eval.png")

## 💾 모델 다운로드 (Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    # 최종 모델 다운로드
    files.download('ppo_policy_best_v5.pt')
    
    print("✅ 모델 다운로드 완료")